# Khám phá kết quả thực nghiệm

Notebook này **đọc lại** các kết quả đã chạy (`reports/`), không huấn luyện lại mô hình.
Chạy `python run.py` trước, rồi mở notebook này để phân tích và chép bảng vào báo cáo.

In [ ]:
import sys, json
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
T = ROOT / 'reports' / 'tables'
pd.set_option('display.width', 200)
print('Thư mục bảng kết quả:', T)

## 1. Chất lượng dữ liệu (đưa vào Chương 3)

In [ ]:
for m in ['vn', 'us']:
    p = T / f'data_quality_{m}.csv'
    if p.exists():
        q = pd.read_csv(p)
        print(f'--- {m.upper()} : {len(q)} mã ---')
        display(q.describe().round(3))
        # Các mã có ngày nhảy giá bất thường (nghi ngờ chia tách / cổ tức chưa điều chỉnh)
        display(q[q['jump_gt_15pct_days'] > 0][['ticker', 'jump_gt_15pct_days', 'ann_vol']])

## 2. Kết quả walk-forward CV (chọn mô hình)

In [ ]:
for m in ['vn', 'us']:
    p = T / f'cv_summary_{m}.csv'
    if p.exists():
        print(f'--- {m.upper()} ---')
        display(pd.read_csv(p))

## 3. Kết quả tập test out-of-sample (Chương 4)

`mcnemar_p` và `dm_p` so sánh từng mô hình với baseline `naive`.
p < 0.05 nghĩa là khác biệt **có ý nghĩa thống kê**.

In [ ]:
for m in ['vn', 'us']:
    p = T / f'test_results_{m}.csv'
    if p.exists():
        print(f'--- {m.upper()} ---')
        display(pd.read_csv(p).round(4))

## 4. Backtest có chi phí giao dịch

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, m in zip(axes, ['vn', 'us']):
    p = T / f'backtest_daily_{m}.csv'
    if not p.exists():
        continue
    d = pd.read_csv(p, index_col=0, parse_dates=True)
    (1 + d['net']).cumprod().plot(ax=ax, label='Chiến lược')
    (1 + d['bh']).cumprod().plot(ax=ax, label='Mua & nắm giữ', alpha=.7)
    ax.set_title(m.upper()); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

In [ ]:
for m in ['vn', 'us']:
    p = T / f'backtest_by_year_{m}.csv'
    if p.exists():
        print(f'--- {m.upper()} ---')
        display(pd.read_csv(p).round(4))

## 5. So sánh hai thị trường (trả lời câu hỏi nghiên cứu)

In [ ]:
comp = pd.read_csv(T / 'market_comparison.csv')
display(comp.round(4))

## 6. Đặc trưng quan trọng nhất (SHAP)

In [ ]:
for m in ['vn', 'us']:
    p = T / f'feature_importance_{m}.csv'
    if p.exists():
        s = pd.read_csv(p, index_col=0).iloc[:, 0].sort_values(ascending=False)
        print(f'--- {m.upper()} : top 15 ---')
        display(s.head(15).round(4))